# Feature Engineering

## CRISP-DM Phase 6: Feature Engineering

### Project: Customer Churn Analysis

Feature Engineering is the process of transforming existing data into meaningful features that can improve the performance, interpretability, and predictive power of machine learning models.

The objectives of this notebook are to:

- Load the cleaned customer churn dataset
- Review insights from the Deep Dive EDA
- Identify relevant features for churn prediction
- Remove irrelevant or redundant features
- Transform numerical and categorical variables into suitable formats
- Create new features from existing customer attributes
- Encode categorical variables appropriately
- Scale numerical features where required
- Handle skewed numerical variables where necessary
- Create customer tenure-based features
- Create customer spending and billing-related features
- Create service usage and engagement-related features
- Create meaningful customer segmentation features
- Analyze the relationship between engineered features and churn
- Check for multicollinearity and redundant features
- Select the most relevant features for modeling
- Validate the final feature set
- Prepare the dataset for machine learning model development

### Input
Cleaned customer churn dataset

### Output
Feature-engineered dataset containing transformed, encoded, and newly created features suitable for machine learning modeling.

### Key Areas of Feature Engineering

- **Feature Creation:** Generate new variables from existing customer information
- **Feature Transformation:** Apply mathematical or logical transformations to improve feature representation
- **Categorical Encoding:** Convert categorical variables into numerical representations
- **Numerical Transformation:** Transform numerical features where required
- **Feature Scaling:** Standardize or normalize numerical variables when appropriate
- **Feature Aggregation:** Combine related variables to create meaningful summary features
- **Customer Tenure Features:** Create meaningful tenure-based groups or indicators
- **Billing Features:** Derive spending, billing, and payment-related features
- **Service Features:** Create features representing the number or type of services used
- **Customer Segmentation:** Create meaningful customer groups based on behavioral characteristics
- **Feature Selection:** Identify and retain features that are relevant to churn prediction
- **Multicollinearity Analysis:** Detect and address highly correlated or redundant features
- **Data Leakage Prevention:** Ensure that engineered features do not use information unavailable at prediction time

### Expected Outcomes

The feature engineering process should help to:

- Improve the quality and representation of the input data
- Capture meaningful customer behavior patterns
- Reduce unnecessary or redundant variables
- Convert categorical information into model-compatible features
- Create features that better explain customer churn
- Improve machine learning model performance
- Produce a reliable and reproducible modeling dataset

### Conclusion

The final feature-engineered dataset will serve as the primary input for machine learning model development. The selected features should capture important customer characteristics, behaviors, service usage, billing patterns, and other factors identified during the EDA phases.


In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/tiyasanin/telco-cutomer-churn/Telco_customer_churn_day.csv
/kaggle/input/notebooks/harshsave/05-deep-dive-customer-churn/__results__.html
/kaggle/input/notebooks/harshsave/05-deep-dive-customer-churn/customer_churn_clean_eda_op.csv
/kaggle/input/notebooks/harshsave/05-deep-dive-customer-churn/__notebook__.ipynb
/kaggle/input/notebooks/harshsave/05-deep-dive-customer-churn/__output__.json
/kaggle/input/notebooks/harshsave/05-deep-dive-customer-churn/custom.css
/kaggle/input/notebooks/harshsave/05-deep-dive-customer-churn/__results___files/__results___8_0.png
/kaggle/input/notebooks/harshsave/05-deep-dive-customer-churn/__results___files/__results___9_0.png
/kaggle/input/notebooks/harshsave/05-deep-dive-customer-churn/__results___files/__results___56_0.png
/kaggle/input/notebooks/harshsave/05-deep-dive-customer-churn/__results___files/__results___27_0.png
/kaggle/input/notebooks/harshsave/05-deep-dive-customer-churn/__results___files/__results___33_0.png
/kaggle/in

# Feature Engineering

## Objective

The objective of this notebook is to perform feature engineering on the cleaned customer churn dataset.

The feature engineering process focuses on identifying meaningful relationships between existing variables and creating new features that may provide additional information for predicting customer churn.

Rather than creating numerous arbitrary features, the approach used in this notebook is **business-driven and hypothesis-based**, ensuring that every engineered feature has a clear interpretation.

---

## 1. Importing Required Libraries and Datasets

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

In [ ]:
engine = create_engine(
    f"postgresql://{DB_CONFIG["user"]}:{DB_CONFIG["password"]}@{DB_CONFIG["host"]}:{DB_CONFIG["port"]}/{DB_CONFIG["database"]}"
)

query = "SELECT * FROM customer_churn_clean_eda_deep_dive;"
customer_churn = pd.read_sql(query, engine)
customer_churn.head()

,Unnamed: 0,customerid,count,country,state,city,zip_code,lat_long,latitude,longitude,...,monthly_charges,total_charges,churn_label,churn_value,churn_score,cltv,churn_reason,is_churned,cltv_band,tenure_band
0,0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,...,53.85,108.15,Yes,1,86,3239,Competitor made better offer,True,low,Very Short Tenure
1,1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,...,70.70,151.65,Yes,1,67,2701,Moved,True,low,Very Short Tenure
2,2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,...,99.65,820.50,Yes,1,86,5372,Moved,True,high,Very Short Tenure
3,3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,...,104.80,3046.05,Yes,1,84,5003,Moved,True,high,Short Tenure
4,4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,...,103.70,5036.30,Yes,1,89,5340,Competitor had better devices,True,high,Long Tenure


In [4]:
df_fe = customer_churn.copy()

## 2. Review Existing Features

Before creating new features, the existing variables were reviewed and grouped according to the type of information they represent.
### Customer Information and Demographics

* CustomerID
* Country
* State
* City
* Zip Code
* Gender
* Senior Citizen
* Partner
* Dependents

### Customer Relationship and Value

* CLTV
* Churn Reason

### Products and Services

* Phone Service
* Internet Service
* Online Security
* Online Backup
* Device Protection
* Tech Support
* Streaming TV
* Streaming Movies

### Account and Billing Information

* Tenure Months
* Contract
* Paperless Billing
* Payment Method
* Monthly Charges
* Total Charges

### Target Variable

* Churn Label

---

## 4. Feature Engineering Approach

Each feature category was reviewed to determine whether meaningful additional information could be derived from the existing variables.

The analysis focused on the following questions:

* Does combining multiple features represent a meaningful business concept?
* Does the engineered feature provide information not clearly represented by the original variables?
* Does the feature have a clear business interpretation?
* Would the information be available at the time of churn prediction?

Most features were retained in their original form because creating additional transformations would not provide a clear business advantage.

Two groups of service-related features were identified as meaningful candidates for feature engineering:

1. Protection and Support Services
2. Entertainment Services

---

# 5. Protection and Support Service Count

The following services provide protection, backup, device protection, or technical support to customers:

* Online Security
* Online Backup
* Device Protection
* Tech Support

Instead of considering only these services individually, a new feature is created to represent the number of protection and support services used by each customer.

The feature is named:

```text
protection_support_count
```

A higher value indicates that the customer has adopted a greater number of protection and support-related services.

The resulting feature can have values ranging from:

```text
0 → No protection or support services
1 → One protection or support service
2 → Two protection or support services
3 → Three protection or support services
4 → Four protection or support services
```

---

In [5]:
protection_support_columns = [
    'online_security', 'online_backup', 'device_protection', 'tech_support'
]
df_fe['protection_support_count'] = (
    
    df_fe[protection_support_columns]
    .eq('Yes')
    .sum(axis = 1)
)

In [6]:
df_fe.head()

,Unnamed: 0,customerid,count,country,state,city,zip_code,lat_long,latitude,longitude,...,total_charges,churn_label,churn_value,churn_score,cltv,churn_reason,is_churned,cltv_band,tenure_band,protection_support_count
0,0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,...,108.15,Yes,1,86,3239,Competitor made better offer,True,low,Very Short Tenure,2
1,1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,...,151.65,Yes,1,67,2701,Moved,True,low,Very Short Tenure,0
2,2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,...,820.50,Yes,1,86,5372,Moved,True,high,Very Short Tenure,1
3,3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,...,3046.05,Yes,1,84,5003,Moved,True,high,Short Tenure,2
4,4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,...,5036.30,Yes,1,89,5340,Competitor had better devices,True,high,Long Tenure,2


## 6. Entertainment Service Count

The dataset contains two entertainment-related services:

* Streaming TV
* Streaming Movies

A new feature is created to represent the number of entertainment services used by each customer.

The feature is named:

```text
entertainment_count
```
```

The resulting feature can have values ranging from:

```text
0 → No entertainment services
1 → One entertainment service
2 → Both entertainment services
```

---

In [7]:
entertainment_columns = [
    'streaming_tv', 
    'streaming_movies'
]

df_fe['entertainment_count'] = (
    df_fe[entertainment_columns]
    .eq('Yes')
    .sum(axis=1)
)

In [8]:
df_fe.head()

,Unnamed: 0,customerid,count,country,state,city,zip_code,lat_long,latitude,longitude,...,churn_label,churn_value,churn_score,cltv,churn_reason,is_churned,cltv_band,tenure_band,protection_support_count,entertainment_count
0,0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,...,Yes,1,86,3239,Competitor made better offer,True,low,Very Short Tenure,2,0
1,1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,...,Yes,1,67,2701,Moved,True,low,Very Short Tenure,0,0
2,2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,...,Yes,1,86,5372,Moved,True,high,Very Short Tenure,1,2
3,3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,...,Yes,1,84,5003,Moved,True,high,Short Tenure,2,2
4,4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,...,Yes,1,89,5340,Competitor had better devices,True,high,Long Tenure,2,2


## 7. Review the Engineered Features

Review the newly created features to ensure they were generated correctly.

In [9]:
df_fe['protection_support_count'].value_counts().sort_index()

protection_support_count
0    2793
1    1467
2    1372
3     941
4     470
Name: count, dtype: int64

In [10]:
df_fe['entertainment_count'].value_counts().sort_index()

entertainment_count
0    3544
1    1559
2    1940
Name: count, dtype: int64

# 8. Review the Final Feature-Engineered Dataset

Check the structure and data types of the updated dataset.

In [11]:
df_fe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 39 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Unnamed: 0                7043 non-null   int64  
 1   customerid                7043 non-null   object 
 2   count                     7043 non-null   int64  
 3   country                   7043 non-null   object 
 4   state                     7043 non-null   object 
 5   city                      7043 non-null   object 
 6   zip_code                  7043 non-null   int64  
 7   lat_long                  7043 non-null   object 
 8   latitude                  7043 non-null   float64
 9   longitude                 7043 non-null   float64
 10  gender                    7043 non-null   object 
 11  senior_citizen            7043 non-null   object 
 12  partner                   7043 non-null   object 
 13  dependents                7043 non-null   object 
 14  tenure_m

In [12]:
df_fe.shape

(7043, 39)

In [13]:
df_fe.isnull().sum()

Unnamed: 0                  0
customerid                  0
count                       0
country                     0
state                       0
city                        0
zip_code                    0
lat_long                    0
latitude                    0
longitude                   0
gender                      0
senior_citizen              0
partner                     0
dependents                  0
tenure_months               0
phone_service               0
multiple_lines              0
internet_service            0
online_security             0
online_backup               0
device_protection           0
tech_support                0
streaming_tv                0
streaming_movies            0
contract                    0
paperless_billing           0
payment_method              0
monthly_charges             0
total_charges               0
churn_label                 0
churn_value                 0
churn_score                 0
cltv                        0
churn_reas

## 9. Feature Engineering Summary

The feature engineering process followed a **minimal and business-driven approach**.

Rather than creating arbitrary transformations or combinations of variables, new features were created only when multiple existing variables represented a meaningful underlying concept.

The following features were created:

### Protection and Support Service Count

```text
protection_support_count
```

This feature represents the number of protection and support-related services adopted by a customer.

### Entertainment Service Count

```text
entertainment_count
```

This feature represents the number of entertainment-related services adopted by a customer.

All other features were retained in their original form because no additional transformation was identified that provided a clear and meaningful business interpretation.

In [ ]:
# Processed Data Path
PROCESSED_DATA_PATH = Path("data/processed/customer_churn_clean_Feature_Engineering.csv")
df_fe.to_csv(PROCESSED_DATA_PATH)

In [ ]:
# Saving the cleaned dataset to Postgresql database
df_fe.to_sql(
    "customer_churn_clean_eda_fe",
    engine,
    if_exists="replace",
    index=False
)